# Visual Relations

Solution author: Cowile

textual description of the solution:

The first thing to notice is that this is a multi-label task: a pair of objects can have multiple predicates simultaneously (a person can be both "riding" and "on" a horse), so you need BCE on each individual logit, not softmax over predicates.

DETR's decoder has 6 layers of self-attention between object queries, each with 8 heads. For any pair of detected objects (i, j), the attention weight from query i to query j tells you how much object i "looks at" object j when refining its representation, and this is directional, the reverse attention carries different information. You extract both directions across all layers and heads (6 x 8 x 2 = 96 features), concatenate the final hidden states of both queries (2 x 256 = 512), and add 6 spatial geometry features (center offsets, log size ratios, IoU, log area ratio) to get a 614-dim feature vector per ordered pair. A small two-head MLP then classifies predicates from that: one head for the 35 predicates (multi-label), one binary head for whether the pair has any relationship at all. You also need to sample negative pairs (pairs with no ground truth relationship) at roughly 2:1 ratio during training, otherwise the model never learns to say "no relation." The core idea of mining relational signal from transformer attention comes from [EGTR: Extracting Graph from Transformer for Scene Graph Generation](https://arxiv.org/abs/2404.02072), though that paper trains on Visual Genome's ~200k images and its full method does not transfer well to our scale of ~6k images.

The second thing to find is the long-tail problem. "on" has about 3000 training examples, "chasing" has 30, and since the problem uses mAP both classes contribute equally to the final score. The fix has two parts. During training, use class-balanced positive weights in the BCE loss based on [Class-Balanced Loss Based on Effective Number of Samples](https://arxiv.org/abs/1901.05555) (Cui et al., CVPR 2019), which down-weights the head classes. At inference, apply [Long-Tail Learning via Logit Adjustment](https://arxiv.org/abs/2007.07314): subtract tau * log(class_frequency) from each predicate's logit before sigmoid, which shifts the decision boundary so rare predicates need lower raw activation to fire. Without both of these, the model basically only predicts "on" and "near" and ignores the tail.

In [ ]:
import random, json, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from torchvision.ops import box_iou
from transformers import DetrImageProcessor, DetrForObjectDetection

seed = 2026
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

BASE = Path('/kaggle/input/competitions/visual-relations-aicc-round-5') # Change to your own path
DEVICE = 'cuda'
DET_THRESHOLD = 0.5
MAX_DETECTIONS = 20

predicates = json.loads((BASE / 'predicates.json').read_text())
pred_to_idx = {p: i for i, p in enumerate(predicates)}
NUM_PREDS = len(predicates)

train_df = pd.read_csv(BASE / 'train_annotations.csv')
train_images = pd.read_csv(BASE / 'train_images.csv')['image_id'].tolist()
test_images = pd.read_csv(BASE / 'test_images.csv')['image_id'].tolist()

2026-03-27 17:10:36.047579: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774631436.253428      30 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774631436.316096      30 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774631436.803008      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774631436.803047      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774631436.803050      30 computation_placer.cc:177] computation placer alr

In [2]:
processor = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')
model = DetrForObjectDetection.from_pretrained('facebook/detr-resnet-50').eval().to(DEVICE)

preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:
def extract_features(image):
    inputs = processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model(**inputs, output_attentions=True)

    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)
    det = processor.post_process_object_detection(out, target_sizes=target_sizes, threshold=DET_THRESHOLD)[0]
    boxes, scores = det['boxes'].cpu(), det['scores'].cpu()

    if len(boxes) < 2:
        return None
    if len(boxes) > MAX_DETECTIONS:
        top = scores.argsort(descending=True)[:MAX_DETECTIONS]
        boxes, scores = boxes[top], scores[top]

    W, H = image.size
    det_norm = torch.stack([
        (boxes[:, 0] + boxes[:, 2]) / 2 / W,
        (boxes[:, 1] + boxes[:, 3]) / 2 / H,
        (boxes[:, 2] - boxes[:, 0]) / W,
        (boxes[:, 3] - boxes[:, 1]) / H,
    ], dim=1)
    qidx = torch.cdist(det_norm, out.pred_boxes[0].cpu()).argmin(dim=1)

    attn = [out.decoder_attentions[l][0][:, qidx][:, :, qidx].cpu() for l in range(6)]
    hidden = out.last_hidden_state[0, qidx].cpu()

    return {'boxes': boxes, 'attn': attn, 'hidden': hidden}

In [4]:
features = {}
for img_id in tqdm(train_images + test_images, desc='Extracting'):
    image = Image.open(BASE / 'images' / f'{img_id}.jpg').convert('RGB')
    feat = extract_features(image)
    if feat is not None:
        features[img_id] = feat

n_train = sum(1 for i in train_images if i in features)
n_test = sum(1 for i in test_images if i in features)
print(f'{n_train}/{len(train_images)} train, {n_test}/{len(test_images)} test')

Extracting: 100%|██████████| 5000/5000 [11:19<00:00,  7.36it/s]

4000/4000 train, 1000/1000 test


In [5]:
def build_pair_features(feat, si, sj):
    parts = []
    for a in feat['attn']:
        parts.append(a[:, si, sj].T)
        parts.append(a[:, sj, si].T)

    parts.extend([feat['hidden'][si], feat['hidden'][sj]])

    bi, bj = feat['boxes'][si], feat['boxes'][sj]
    wi = (bi[:, 2] - bi[:, 0]).clamp(min=1)
    hi = (bi[:, 3] - bi[:, 1]).clamp(min=1)
    wj = (bj[:, 2] - bj[:, 0]).clamp(min=1)
    hj = (bj[:, 3] - bj[:, 1]).clamp(min=1)
    cx_i, cy_i = (bi[:, 0] + bi[:, 2]) / 2, (bi[:, 1] + bi[:, 3]) / 2
    cx_j, cy_j = (bj[:, 0] + bj[:, 2]) / 2, (bj[:, 1] + bj[:, 3]) / 2

    inter_w = (torch.min(bi[:, 2], bj[:, 2]) - torch.max(bi[:, 0], bj[:, 0])).clamp(min=0)
    inter_h = (torch.min(bi[:, 3], bj[:, 3]) - torch.max(bi[:, 1], bj[:, 1])).clamp(min=0)
    iou = inter_w * inter_h / (wi * hi + wj * hj - inter_w * inter_h + 1e-6)

    parts.append(torch.stack([
        (cx_j - cx_i) / wi, (cy_j - cy_i) / hi,
        (wj / wi).log(), (hj / hi).log(),
        iou, (wi * hi / (wj * hj)).log(),
    ], dim=1))

    return torch.cat(parts, dim=1)

In [6]:
X_parts, y_parts = [], []
grouped = train_df.groupby('image_id')

for img_id in tqdm(train_images, desc='Building pairs'):
    if img_id not in features or img_id not in grouped.groups:
        continue
    feat = features[img_id]
    boxes, n = feat['boxes'], len(feat['boxes'])

    gt = grouped.get_group(img_id)
    gt_sub = torch.tensor(gt[['subject_x1', 'subject_y1', 'subject_x2', 'subject_y2']].values, dtype=torch.float32)
    gt_obj = torch.tensor(gt[['object_x1', 'object_y1', 'object_x2', 'object_y2']].values, dtype=torch.float32)
    sub_ious = box_iou(boxes, gt_sub)
    obj_ious = box_iou(boxes, gt_obj)
    best_sub_iou, best_sub = sub_ious.max(dim=0)
    best_obj_iou, best_obj = obj_ious.max(dim=0)

    positives = {}
    pred_names = gt['predicate'].tolist()
    for g in range(len(gt)):
        si, oi = best_sub[g].item(), best_obj[g].item()
        if si == oi or best_sub_iou[g] < 0.5 or best_obj_iou[g] < 0.5:
            continue
        pidx = pred_to_idx.get(pred_names[g])
        if pidx is None:
            continue
        key = (si, oi)
        if key not in positives:
            positives[key] = torch.zeros(NUM_PREDS)
        positives[key][pidx] = 1

    if positives:
        pos_i = torch.tensor([k[0] for k in positives], dtype=torch.long)
        pos_j = torch.tensor([k[1] for k in positives], dtype=torch.long)
        X_parts.append(build_pair_features(feat, pos_i, pos_j))
        y_parts.append(torch.stack(list(positives.values())))

    neg_pairs = [(i, j) for i in range(n) for j in range(n) if i != j and (i, j) not in positives]
    random.shuffle(neg_pairs)
    neg_pairs = neg_pairs[:len(positives) * 2]
    if neg_pairs:
        neg_i = torch.tensor([p[0] for p in neg_pairs], dtype=torch.long)
        neg_j = torch.tensor([p[1] for p in neg_pairs], dtype=torch.long)
        X_parts.append(build_pair_features(feat, neg_i, neg_j))
        y_parts.append(torch.zeros(len(neg_pairs), NUM_PREDS))

X = torch.cat(X_parts)
y = torch.cat(y_parts)
n_pos = int(y.any(dim=1).sum())
print(f'{len(X)} pairs ({n_pos} pos, {len(X) - n_pos} neg), dim={X.shape[1]}')

Building pairs: 100%|██████████| 4000/4000 [00:11<00:00, 339.23it/s]


26946 pairs (9214 pos, 17732 neg), dim=614


In [7]:
class RelHead(nn.Module):
    def __init__(self, dim_in, n_preds):
        super().__init__()
        self.pred = nn.Sequential(
            nn.Linear(dim_in, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, n_preds),
        )
        self.conn = nn.Sequential(
            nn.Linear(dim_in, 256), nn.ReLU(), nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.pred(x), self.conn(x).squeeze(-1)


net = RelHead(X.shape[1], NUM_PREDS).to(DEVICE)

counts = y.sum(0).numpy()
beta = 0.999
eff = np.where(counts > 0, (1 - beta ** counts) / (1 - beta), 1.0)
w = 1.0 / (eff + 1e-8)
pos_weight = torch.tensor(w / w.sum() * NUM_PREDS, dtype=torch.float32).to(DEVICE)

has_rel = y.any(dim=1).float()
loader = DataLoader(TensorDataset(X, y, has_rel), batch_size=256, shuffle=True)
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40)

for ep in range(40):
    net.train()
    total = 0
    for xb, yb, cb in loader:
        xb, yb, cb = xb.to(DEVICE), yb.to(DEVICE), cb.to(DEVICE)
        rl, cl = net(xb)
        loss_r = F.binary_cross_entropy_with_logits(rl, yb, pos_weight=pos_weight)
        loss_c = F.binary_cross_entropy_with_logits(cl, cb)
        loss = loss_r + 0.5 * loss_c
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    sched.step()
    if (ep + 1) % 10 == 0:
        print(f'epoch {ep+1}/40  loss={total / len(X):.4f}')

epoch 10/40  loss=0.1315
epoch 20/40  loss=0.0940
epoch 30/40  loss=0.0712
epoch 40/40  loss=0.0649


In [8]:
TAU = 0.5
freq = torch.zeros(NUM_PREDS)
for p, c in train_df['predicate'].value_counts().items():
    if p in pred_to_idx:
        freq[pred_to_idx[p]] = c
log_freq = torch.log(freq / freq.sum() + 1e-8).to(DEVICE)

net.eval()
results = {}

for img_id in tqdm(test_images, desc='Predicting'):
    if img_id not in features:
        results[img_id] = []
        continue

    feat = features[img_id]
    boxes = feat['boxes']
    n = len(boxes)

    si, sj = torch.where(~torch.eye(n, dtype=bool))
    pf = build_pair_features(feat, si, sj)

    with torch.no_grad():
        rl, cl = net(pf.to(DEVICE))
        rp = torch.sigmoid(rl - TAU * log_freq).cpu()
        cp = torch.sigmoid(cl).cpu()

    scores = rp * cp.unsqueeze(1)
    scores[cp < 0.3] = 0

    pi, ki = torch.where(scores > 0.02)
    vals = scores[pi, ki]
    order = vals.argsort(descending=True)[:100]

    results[img_id] = [
        (predicates[ki[m].item()], vals[m].item(),
         boxes[si[pi[m]]].tolist(), boxes[sj[pi[m]]].tolist())
        for m in order
    ]

Predicting: 100%|██████████| 1000/1000 [00:03<00:00, 283.75it/s]


## Save submission

In [9]:
rows = []
for img_id in test_images:
    ps = results.get(img_id, [])
    parts = [f'{name} {conf:.4f} {sb[0]:.1f} {sb[1]:.1f} {sb[2]:.1f} {sb[3]:.1f} '
             f'{ob[0]:.1f} {ob[1]:.1f} {ob[2]:.1f} {ob[3]:.1f}' for name, conf, sb, ob in ps]
    rows.append({'image_id': img_id, 'PredictionString': ' '.join(parts) if parts else ' '})

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
print(f'Saved {len(submission)} rows, {(submission.PredictionString.str.strip().str.len() > 0).sum()} non-empty')

Saved 1000 rows, 997 non-empty


this solution achieved a score of 0.17695